# LOFAR + DEMON Hydrophone Analysis Pipeline (v5)
## Two-Stage Vessel Detection with Conditional DEMON Blade-Rate Extraction
### NUWR Goa — 512 Hz Hydrophone Recordings

**Input:** 1-hour WAV files from seabed hydrophone (512 Hz sample rate)  
**Output:** Per-window CSV with LOFAR features + conditional DEMON blade-rate/shaft-rate, event list, spectrograms, customer-format CSV

---

### Pipeline Architecture

| Stage | Purpose | Method | Output |
|-------|---------|--------|--------|
| **Stage 1 — Detection** | Is something there? | RMS energy per 30s window, σ-threshold | Detection flag, energy level |
| **Stage 2 — Classification** | What does it look like? | STFT → background normalisation → tonal detection | Tonal frequencies, spectral fingerprint |
| **Stage 3 — Characterisation** | Propeller parameters (conditional) | Hilbert envelope → DEMON spectrum, LOFAR-informed bandpass | Blade rate, shaft rate, RPM, confidence flag |

**Stage 3 runs ONLY on windows where Stage 1 confirms elevated energy AND Stage 2 confirms tonal signatures.**  
At 512 Hz, DEMON operates on low-frequency machinery noise (not cavitation — that requires 20–50 kHz).  
Results are flagged as HIGH (harmonics confirmed, strong RMS, multiple tonals) / LOW / NONE confidence.

**Processing time:** ~30 min for 48 hours (Stages 1+2: ~25 min, Stage 3 conditional: ~5 min)

**Note:** Ocean swell frequencies (0.05–0.1 Hz) and microseisms (0.1–1.0 Hz) are well below
the DEMON blade-rate search range (2–50 Hz), so no swell-band exclusion is needed.


## 1. Setup & Imports

In [ ]:
!pip install soundfile -q

import numpy as np
import soundfile as sf
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.signal import spectrogram as scipy_spectrogram
from scipy.signal import welch, find_peaks, hilbert, butter, sosfilt
from datetime import datetime, timedelta
from pathlib import Path
import os, re, json, warnings, time as _time, gc
warnings.filterwarnings('ignore')

print('All imports OK (including scipy.signal.hilbert, butter, sosfilt for DEMON)')


All imports OK (including scipy.signal.hilbert, butter, sosfilt for DEMON)


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


Mounted at /content/drive
Drive mounted


## 3. Configuration

Edit this cell to match your deployment.


In [ ]:
# =====================================================================
# DATA PATHS — Recording 3: 07–09 Feb 2026 (days 11–13 of trial)
# =====================================================================
WAV_DIRS = [
    '/content/drive/MyDrive/SKANN_SSL/authorised_sessions/',
]

# Sessions to process (set to None for all files in WAV_DIRS)
KEEP_SESSIONS = {'28jan_1', '28jan_2', '28jan_3', '28jan_4'}

# Output directory
OUTPUT_DIR = '/content/drive/MyDrive/SKANN_SSL/lofar_results_28jan/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================================
# SPECTROGRAM PARAMETERS (Stage 2)
# =====================================================================
STFT_WINDOW_SEC = 4.0       # seconds → 0.25 Hz native resolution
STFT_OVERLAP_FRAC = 0.75    # 75% overlap → ~1 spectrum/second
STFT_NFFT_MULT = 2          # zero-pad factor → 0.125 Hz bins
FREQ_MAX_HZ = 256           # display/analysis up to Nyquist

# =====================================================================
# ANALYSIS WINDOWS (for CSV output)
# =====================================================================
ANALYSIS_WINDOW_SEC = 30     # seconds per output row
ANALYSIS_HOP_SEC = 15        # hop → 50% overlap

# =====================================================================
# BACKGROUND NORMALISATION (Stage 2)
# =====================================================================
BACKGROUND_PERCENTILE = 50   # median

# =====================================================================
# TONAL DETECTION (Stage 2)
# =====================================================================
TONAL_THRESHOLD_DB = 8.0     # dB above normalised background to count as tonal
TONAL_MIN_PERSIST_SEC = 30   # must persist this long to be "real"
TONAL_FREQ_MIN_HZ = 3.5     # ignore below this (low-frequency noise)
TONAL_FREQ_MAX_HZ = 256     # upper limit
TONAL_PEAK_PROMINENCE_DB = 3.0
TONAL_PEAK_MIN_DISTANCE_HZ = 1.0

# =====================================================================
# EVENT DETECTION (Stage 1)
# =====================================================================
RMS_SIGMA_THRESHOLD = 1.0    # σ above mean for elevated energy
EVENT_MERGE_RADIUS = 8       # ±8 windows for temporal merging (120s)
EVENT_MIN_WINDOWS = 5        # minimum windows per event
SIGNIFICANT_DURATION_MIN = 5 # minutes for "significant" transit
SIGNIFICANT_RMS_FACTOR = 5.0 # ×background for "significant" transit

# =====================================================================
# DEMON PARAMETERS (Stage 3 — conditional)
# =====================================================================
# Stage 3 only runs on windows where:
#   - RMS energy > DEMON_RMS_GATE × global mean (Stage 1 confirms vessel)
#   - n_tonals >= DEMON_MIN_TONALS (Stage 2 confirms tonal structure)
# Otherwise blade_rate and shaft_rate are left blank.
DEMON_RMS_GATE = 3.0         # ×mean RMS to trigger DEMON
DEMON_MIN_TONALS = 2         # minimum tonals from Stage 2
DEMON_BLADE_RATE_MIN = 2.0   # Hz — lower bound of blade-rate search
DEMON_BLADE_RATE_MAX = 50.0  # Hz — upper search limit
DEMON_FFT_PAD = 4            # zero-pad factor for envelope FFT
DEMON_FILTER_ORDER = 4       # Butterworth bandpass order
DEMON_FALLBACK_BP = (20, 200)  # Hz — fallback bandpass if no LOFAR guidance

# =====================================================================
# SPECTROGRAM IMAGE GENERATION
# =====================================================================
GENERATE_FILE_SPECTROGRAMS = True
GENERATE_EVENT_SPECTROGRAMS = True
EVENT_PADDING_MIN = 5
EVENT_SPEC_MIN_RMS_FACTOR = 10.0

# =====================================================================
# CHECKPOINT
# =====================================================================
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, '_checkpoint.json')

# =====================================================================
# PLOT STYLE
# =====================================================================
STYLE = {
    'bg': '#0a1628', 'text': '#caf0f8', 'accent': '#00b4d8',
    'highlight': '#ff6b35', 'green': '#2ecc71', 'red': '#e74c3c',
    'box_fill': '#0d2137', 'box_edge': '#234567',
    'cmap': 'inferno',
}

print('Configuration loaded')
print(f'  STFT: {STFT_WINDOW_SEC}s window, {STFT_OVERLAP_FRAC*100:.0f}% overlap, {STFT_NFFT_MULT}x zero-pad')
print(f'  Analysis windows: {ANALYSIS_WINDOW_SEC}s / {ANALYSIS_HOP_SEC}s hop')
print(f'  Tonal detection: >{TONAL_THRESHOLD_DB} dB, >{TONAL_MIN_PERSIST_SEC}s, {TONAL_FREQ_MIN_HZ}–{TONAL_FREQ_MAX_HZ} Hz')
print(f'  Event detection: {RMS_SIGMA_THRESHOLD}σ RMS, ±{EVENT_MERGE_RADIUS} merge')
print(f'  DEMON gating: RMS >{DEMON_RMS_GATE}× mean AND ≥{DEMON_MIN_TONALS} tonals')
print(f'  DEMON blade-rate range: {DEMON_BLADE_RATE_MIN}–{DEMON_BLADE_RATE_MAX} Hz')
print(f'  Sessions: {sorted(KEEP_SESSIONS)}' if KEEP_SESSIONS else '  Sessions: ALL')
print(f'  Output: {OUTPUT_DIR}')


Configuration loaded
  STFT: 4.0s window, 75% overlap, 2x zero-pad
  Analysis windows: 30s / 15s hop
  Tonal detection: >8.0 dB, >30s, 3.5–256 Hz
  Event detection: 1.0σ RMS, ±8 merge
  DEMON gating: RMS >3.0× mean AND ≥2 tonals
  DEMON blade-rate range: 2.0–50.0 Hz
  Sessions: ['28jan_1', '28jan_2', '28jan_3', '28jan_4']
  Output: /content/drive/MyDrive/SKANN_SSL/lofar_results_28jan/


## 4. Discover & Sort WAV Files

In [ ]:
def parse_filename(fname):
    """Extract session ID and start timestamp from filename.
    Handles both formats:
      session_N_YYYY-MM-DD_HH-MM-SS_to_...       (numeric sessions)
      session_28jan_N_YYYY-MM-DD_HH-MM-SS_to_... (28 Jan sessions)
    Returns session_id as STRING in both cases to avoid mixed-type sort errors.
    """
    # 28jan format — check first (more specific pattern)
    m = re.match(r'session_(28jan_\d+)_(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})_to_', fname)
    if m:
        session_id = m.group(1)          # e.g. '28jan_1'
        ts = datetime.strptime(f'{m.group(2)}_{m.group(3)}', '%Y-%m-%d_%H-%M-%S')
        return session_id, ts

    # Standard numeric format — return as string to keep types uniform
    m = re.match(r'session_(\d+)_(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})_to_', fname)
    if m:
        session_id = m.group(1)          # e.g. '1', '2', ... (string, not int)
        ts = datetime.strptime(f'{m.group(2)}_{m.group(3)}', '%Y-%m-%d_%H-%M-%S')
        return session_id, ts

    return None, None

# Gather all WAV files
all_files = []
for d in WAV_DIRS:
    if not os.path.exists(d):
        print(f'WARNING: {d} not found')
        continue
    for f in sorted(os.listdir(d)):
        if not f.endswith('.wav'):
            continue
        session, ts = parse_filename(f)
        if session is None:
            continue
        if KEEP_SESSIONS and session not in KEEP_SESSIONS:
            continue
        all_files.append({
            'filename': f,
            'path': os.path.join(d, f),
            'session': session,
            'start_ts': ts,
        })

all_files = sorted(all_files, key=lambda x: x['start_ts'])

print(f'Found {len(all_files)} WAV files to process')
if all_files:
    print(f'  First: session {all_files[0]["session"]} — {all_files[0]["start_ts"]}')
    print(f'  Last:  session {all_files[-1]["session"]} — {all_files[-1]["start_ts"]}')
    total_hours = (all_files[-1]['start_ts'] - all_files[0]['start_ts']).total_seconds() / 3600 + 1
    print(f'  Coverage: ~{total_hours:.0f} hours')


Found 4 WAV files to process
  First: session 28jan_1 — 2026-01-28 13:25:19
  Last:  session 28jan_4 — 2026-01-28 16:25:18
  Coverage: ~4 hours


## 5. Processing Functions

### Stages 1 & 2: RMS Energy + LOFAR Tonal Detection (unchanged from v3)
### Stage 3: DEMON Blade-Rate Extraction (NEW — conditional on Stages 1+2)


In [ ]:
# =====================================================================
# STAGE 2: LOFAR FUNCTIONS (unchanged from v3)
# =====================================================================

def compute_stft(audio, sr):
    """Compute STFT spectrogram.
    Returns freqs (Hz), times (seconds), Sxx (power spectral density)."""
    nperseg = int(STFT_WINDOW_SEC * sr)
    noverlap = int(nperseg * STFT_OVERLAP_FRAC)
    nfft = nperseg * STFT_NFFT_MULT

    freqs, times, Sxx = scipy_spectrogram(
        audio, fs=sr, nperseg=nperseg, noverlap=noverlap,
        nfft=nfft, mode='psd', scaling='density'
    )
    return freqs, times, Sxx


def normalise_spectrogram(freqs, times, Sxx):
    """Normalise spectrogram against background (median spectrum).
    Returns normalised spectrogram in dB above background."""
    background = np.percentile(Sxx, BACKGROUND_PERCENTILE, axis=1, keepdims=True)
    background = np.maximum(background, 1e-20)
    spec_norm = Sxx / background
    spec_db = 10 * np.log10(np.maximum(spec_norm, 1e-10))
    return spec_db, background.squeeze()


def detect_tonals_in_window(spec_db_window, freqs):
    """Detect tonal peaks in a single normalised spectrum.
    Returns list of {freq_hz, strength_db, prominence_db}."""
    mask = (freqs >= TONAL_FREQ_MIN_HZ) & (freqs <= TONAL_FREQ_MAX_HZ)
    f_sub = freqs[mask]
    s_sub = spec_db_window[mask]

    if len(s_sub) < 10:
        return []

    freq_res = freqs[1] - freqs[0] if len(freqs) > 1 else 0.125
    min_distance = max(1, int(TONAL_PEAK_MIN_DISTANCE_HZ / freq_res))

    peaks, properties = find_peaks(
        s_sub,
        height=TONAL_THRESHOLD_DB,
        prominence=TONAL_PEAK_PROMINENCE_DB,
        distance=min_distance,
    )

    tonals = []
    for i, pk in enumerate(peaks):
        tonals.append({
            'freq_hz': float(f_sub[pk]),
            'strength_db': float(s_sub[pk]),
            'prominence_db': float(properties['prominences'][i]),
        })

    return sorted(tonals, key=lambda t: t['strength_db'], reverse=True)


def detect_persistent_tonals(spec_db, freqs, times):
    """Track tonals across time to find persistent narrowband lines.
    Groups into 2 Hz bands, requires sustained presence."""
    freq_mask = (freqs >= TONAL_FREQ_MIN_HZ) & (freqs <= TONAL_FREQ_MAX_HZ)
    f_sub = freqs[freq_mask]
    s_sub = spec_db[freq_mask, :]

    if len(times) < 2 or len(f_sub) < 2:
        return []

    dt = times[1] - times[0]
    min_frames = max(1, int(TONAL_MIN_PERSIST_SEC / dt))
    freq_res = freqs[1] - freqs[0]

    band_width_hz = 2.0
    band_width_bins = max(1, int(band_width_hz / freq_res))
    n_bands = len(f_sub) // band_width_bins
    band_freqs = []
    band_spec = np.zeros((n_bands, s_sub.shape[1]))
    for bi in range(n_bands):
        start_bin = bi * band_width_bins
        end_bin = min(start_bin + band_width_bins, len(f_sub))
        band_spec[bi, :] = np.max(s_sub[start_bin:end_bin, :], axis=0)
        peak_bin = start_bin + np.argmax(np.mean(s_sub[start_bin:end_bin, :], axis=1))
        band_freqs.append(float(f_sub[peak_bin]))

    above = band_spec > TONAL_THRESHOLD_DB
    persistent = []

    for bi in range(n_bands):
        row = above[bi, :]
        start_idx = None
        for ti in range(len(row)):
            if row[ti] and start_idx is None:
                start_idx = ti
            elif not row[ti] and start_idx is not None:
                if ti - start_idx >= min_frames:
                    persistent.append({
                        'freq_hz': band_freqs[bi],
                        'start_sec': float(times[start_idx]),
                        'end_sec': float(times[ti - 1]),
                        'duration_sec': float(times[ti - 1] - times[start_idx]),
                        'mean_db': float(np.mean(band_spec[bi, start_idx:ti])),
                        'max_db': float(np.max(band_spec[bi, start_idx:ti])),
                    })
                start_idx = None
        if start_idx is not None and len(row) - start_idx >= min_frames:
            persistent.append({
                'freq_hz': band_freqs[bi],
                'start_sec': float(times[start_idx]),
                'end_sec': float(times[-1]),
                'duration_sec': float(times[-1] - times[start_idx]),
                'mean_db': float(np.mean(band_spec[bi, start_idx:])),
                'max_db': float(np.max(band_spec[bi, start_idx:])),
            })

    return sorted(persistent, key=lambda t: t['mean_db'], reverse=True)


# =====================================================================
# STAGE 3: DEMON BLADE-RATE EXTRACTION (NEW)
# =====================================================================
# Runs ONLY when Stage 1 (RMS) and Stage 2 (tonals) confirm a vessel.
# Bandpass selection is informed by LOFAR tonal detections — no PSO needed.
#
# At 512 Hz, this operates on low-frequency machinery noise, NOT cavitation
# (which requires 20–50 kHz). Results are flagged with confidence accordingly.
# =====================================================================

def _design_demon_bandpass(tonal_freqs_hz, sr):
    """Design a bandpass filter informed by LOFAR tonal detections.

    Strategy:
    - If LOFAR found tonals: centre the bandpass on the strongest tonal cluster,
      wide enough to capture nearby lines.
    - If no usable tonals: fall back to a wide bandpass (20–200 Hz).

    Returns (sos, low_hz, high_hz) — the filter and its bounds.
    """
    nyquist = sr / 2.0

    if tonal_freqs_hz and len(tonal_freqs_hz) > 0:
        # Use the strongest tonal cluster
        # Find the median frequency of detected tonals as the centre
        freqs = sorted(tonal_freqs_hz)
        centre = np.median(freqs)

        # Bandwidth: span from lowest to highest tonal, plus ±20 Hz padding
        # but at least 40 Hz wide
        span = max(freqs) - min(freqs) if len(freqs) > 1 else 0
        half_bw = max(20, (span / 2) + 20)

        low_hz = max(5.0, centre - half_bw)   # stay above 5 Hz
        high_hz = min(nyquist * 0.9, centre + half_bw)  # stay below Nyquist
    else:
        # Fallback: wide bandpass
        low_hz, high_hz = DEMON_FALLBACK_BP
        high_hz = min(high_hz, nyquist * 0.9)

    # Ensure valid filter range
    if high_hz <= low_hz + 5:
        low_hz, high_hz = DEMON_FALLBACK_BP
        high_hz = min(high_hz, nyquist * 0.9)

    sos = butter(DEMON_FILTER_ORDER, [low_hz, high_hz], btype='band',
                 fs=sr, output='sos')
    return sos, low_hz, high_hz


def compute_demon_features(audio_chunk, sr, tonal_freqs_hz):
    """Run DEMON envelope analysis on a single 30-second audio chunk.

    1. Bandpass filter (informed by LOFAR tonals or fallback wideband)
    2. Hilbert transform → envelope extraction
    3. AC-couple (remove mean)
    4. Zero-padded FFT → modulation spectrum
    5. Peak detection in blade-rate range (2–50 Hz)
    6. Harmonic consistency check (2×, 3× fundamental)

    Returns dict with blade_rate_hz, shaft_rate_hz, est_n_blades,
    demon_score, bandpass used, or None values if no clear peak found.
    """
    # Design bandpass from LOFAR information
    try:
        sos, bp_low, bp_high = _design_demon_bandpass(tonal_freqs_hz, sr)
        filtered = sosfilt(sos, audio_chunk)
    except Exception:
        # Filter design can fail near Nyquist — use fallback
        try:
            low, high = DEMON_FALLBACK_BP
            high = min(high, sr / 2 * 0.9)
            sos = butter(DEMON_FILTER_ORDER, [low, high], btype='band',
                         fs=sr, output='sos')
            filtered = sosfilt(sos, audio_chunk)
            bp_low, bp_high = low, high
        except Exception:
            return _empty_demon_result()

    # Hilbert envelope
    analytic = hilbert(filtered)
    envelope = np.abs(analytic)

    # AC-couple (remove DC)
    envelope_ac = envelope - np.mean(envelope)

    # Zero-padded FFT of envelope → modulation spectrum
    n_fft = len(envelope_ac) * DEMON_FFT_PAD
    fft_result = np.fft.rfft(envelope_ac, n=n_fft)
    mod_freqs = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    mod_spectrum = np.abs(fft_result) / len(envelope_ac)

    # Search for blade-rate peaks in the valid range
    br_mask = (mod_freqs >= DEMON_BLADE_RATE_MIN) & (mod_freqs <= DEMON_BLADE_RATE_MAX)
    if not br_mask.any():
        return _empty_demon_result()

    br_freqs = mod_freqs[br_mask]
    br_spec = mod_spectrum[br_mask]

    # Find peaks with minimum prominence
    freq_res = mod_freqs[1] - mod_freqs[0] if len(mod_freqs) > 1 else 0.01
    min_dist = max(1, int(1.0 / freq_res))  # at least 1 Hz apart

    peaks, props = find_peaks(br_spec, prominence=np.median(br_spec) * 0.5,
                              distance=min_dist)

    if len(peaks) == 0:
        return _empty_demon_result()

    # Take the strongest peak as candidate blade rate
    best_idx = peaks[np.argmax(props['prominences'])]
    blade_rate = float(br_freqs[best_idx])
    blade_prominence = float(props['prominences'][np.argmax(props['prominences'])])

    # Check for harmonics at 2× and 3× blade rate
    harmonic_score = 0
    for mult in [2, 3]:
        target = blade_rate * mult
        if target > DEMON_BLADE_RATE_MAX or target > sr / 2:
            continue
        h_idx = np.argmin(np.abs(mod_freqs - target))
        # Is there a peak near the expected harmonic?
        local_region = mod_spectrum[max(0, h_idx-3):h_idx+4]
        if len(local_region) > 0 and np.max(local_region) > np.median(br_spec) * 1.5:
            harmonic_score += 1

    # Estimate shaft rate and blade count
    # Try common blade counts (3, 4, 5) and see which gives a shaft-rate
    # sub-harmonic that matches a peak in the spectrum
    best_n_blades = 4  # default assumption
    best_shaft_match = 0
    for n_blades in [3, 4, 5]:
        shaft_rate = blade_rate / n_blades
        if shaft_rate < 0.5:
            continue
        s_idx = np.argmin(np.abs(mod_freqs - shaft_rate))
        local_region = mod_spectrum[max(0, s_idx-2):s_idx+3]
        if len(local_region) > 0:
            match_strength = np.max(local_region) / np.median(br_spec)
            if match_strength > best_shaft_match:
                best_shaft_match = match_strength
                best_n_blades = n_blades

    shaft_rate = blade_rate / best_n_blades
    rpm = shaft_rate * 60

    # Compute a simple DEMON quality score
    # (prominence of blade-rate peak + harmonic bonus, no fixed saturation)
    noise_floor = np.median(br_spec)
    snr = blade_prominence / noise_floor if noise_floor > 0 else 0
    demon_score = snr + harmonic_score * 2.0

    return {
        'blade_rate_hz': round(blade_rate, 2),
        'shaft_rate_hz': round(shaft_rate, 3),
        'est_n_blades': best_n_blades,
        'est_rpm': round(rpm, 1),
        'demon_score': round(demon_score, 2),
        'demon_harmonics': harmonic_score,
        'demon_bp_low': round(bp_low, 1),
        'demon_bp_high': round(bp_high, 1),
    }


def _empty_demon_result():
    """Return empty DEMON result dict (no detection)."""
    return {
        'blade_rate_hz': None,
        'shaft_rate_hz': None,
        'est_n_blades': None,
        'est_rpm': None,
        'demon_score': 0.0,
        'demon_harmonics': 0,
        'demon_bp_low': None,
        'demon_bp_high': None,
    }


# =====================================================================
# COMBINED: Process one analysis window (Stages 1 + 2 + conditional 3)
# =====================================================================

def process_analysis_window(audio_chunk, sr, spec_db_chunk, freqs, times_chunk,
                            rms_mean_global=None):
    """Process a single 30-second analysis window through all stages.

    Stage 1: RMS energy (always runs)
    Stage 2: LOFAR tonal detection (always runs)
    Stage 3: DEMON blade-rate (only if Stage 1 + Stage 2 thresholds met)

    rms_mean_global: if provided, used for DEMON gating. If None, Stage 3 skipped.
    """
    # ---- Stage 1: RMS energy ----
    rms = float(np.sqrt(np.mean(audio_chunk ** 2)))

    # ---- Stage 2: LOFAR tonal detection ----
    mean_spec = np.mean(spec_db_chunk, axis=1)
    tonals = detect_tonals_in_window(mean_spec, freqs)
    n_tonals = len(tonals)
    tonal_energy = sum(t['strength_db'] for t in tonals) if tonals else 0.0
    strongest_freq = tonals[0]['freq_hz'] if tonals else 0.0
    strongest_db = tonals[0]['strength_db'] if tonals else 0.0
    tonal_freqs = [t['freq_hz'] for t in tonals[:5]]

    # Band energy
    band_masks = {
        'low_5_20': (freqs >= 5) & (freqs < 20),
        'mid_20_80': (freqs >= 20) & (freqs < 80),
        'high_80_256': (freqs >= 80) & (freqs <= 256),
    }
    band_energy = {}
    for band_name, bmask in band_masks.items():
        if bmask.any():
            band_energy[band_name] = float(np.mean(mean_spec[bmask]))
        else:
            band_energy[band_name] = 0.0

    # Vessel score (raw — recomputed post-hoc in §7 with soft gating)
    vessel_score = (
        n_tonals * 1.0 +
        tonal_energy / 10.0 +
        strongest_db / 3.0
    )

    # ---- Stage 3: DEMON (conditional) ----
    demon_confidence = 'NONE'
    demon = _empty_demon_result()

    if rms_mean_global is not None:
        rms_ratio = rms / rms_mean_global
        if rms_ratio >= DEMON_RMS_GATE and n_tonals >= DEMON_MIN_TONALS:
            # Gate passed — run DEMON with LOFAR-informed bandpass
            demon = compute_demon_features(audio_chunk, sr, tonal_freqs)

            if demon['blade_rate_hz'] is not None:
                # Determine confidence
                if (demon['demon_harmonics'] >= 1
                        and rms_ratio >= 5.0 and n_tonals >= 3):
                    demon_confidence = 'HIGH'
                else:
                    demon_confidence = 'LOW'

    result = {
        'rms_energy': rms,
        'n_tonals': n_tonals,
        'tonal_energy': round(tonal_energy, 2),
        'strongest_tonal_hz': round(strongest_freq, 2),
        'strongest_tonal_db': round(strongest_db, 2),
        'tonal_freqs': '|'.join(f'{f:.1f}' for f in tonal_freqs),
        'band_low_5_20_db': round(band_energy['low_5_20'], 2),
        'band_mid_20_80_db': round(band_energy['mid_20_80'], 2),
        'band_high_80_256_db': round(band_energy['high_80_256'], 2),
        'vessel_score': round(vessel_score, 2),
        # Stage 3 outputs
        'blade_rate_hz': demon['blade_rate_hz'],
        'shaft_rate_hz': demon['shaft_rate_hz'],
        'est_n_blades': demon['est_n_blades'],
        'est_rpm': demon['est_rpm'],
        'demon_score': demon['demon_score'],
        'demon_harmonics': demon['demon_harmonics'],
        'demon_bp_low': demon['demon_bp_low'],
        'demon_bp_high': demon['demon_bp_high'],
        'demon_confidence': demon_confidence,
    }
    return result


print('Processing functions loaded (Stages 1 + 2 + 3)')


Processing functions loaded (Stages 1 + 2 + 3)


## 6. Process All Files

**Two-pass approach:**
1. **Pass 1:** Process all files through Stages 1+2 (RMS + LOFAR). This establishes the global RMS mean needed for Stage 3 gating.
2. **Pass 2:** Re-process elevated-energy windows through Stage 3 (DEMON) using the global RMS mean for gating.


In [ ]:
def process_one_file(file_info, file_idx, total_files, rms_mean_global=None):
    """Process a single WAV file through the pipeline.

    If rms_mean_global is None: Stages 1+2 only (Pass 1).
    If rms_mean_global is provided: Stages 1+2+3 (Pass 2 — full pipeline).
    """
    fname = file_info['filename']
    fpath = file_info['path']
    file_start = file_info['start_ts']
    session = file_info['session']

    mode = 'Stages 1+2+3' if rms_mean_global else 'Stages 1+2'
    print(f'  [{file_idx+1}/{total_files}] session_{session} — {file_start.strftime("%d %b %H:%M")} ({mode})')

    # Load audio
    try:
        audio, sr = sf.read(fpath)
    except Exception as e:
        print(f'    ERROR reading file: {e}')
        return []
    if audio.ndim > 1:
        audio = audio[:, 0]
    duration_sec = len(audio) / sr

    print(f'    {sr} Hz, {duration_sec:.0f}s, {len(audio)/1e6:.1f}M samples')

    # ---- STFT spectrogram for full file ----
    freqs, stft_times, Sxx = compute_stft(audio, sr)
    spec_db, background = normalise_spectrogram(freqs, stft_times, Sxx)

    # ---- Optional: save full-file LOFAR spectrogram image ----
    if GENERATE_FILE_SPECTROGRAMS and rms_mean_global is not None:
        import matplotlib.dates as mdates
        from matplotlib.dates import DateFormatter as _DF, MinuteLocator as _ML

        stft_dts = np.array(
            [file_start + timedelta(seconds=float(t)) for t in stft_times],
            dtype='datetime64[ms]'
        ).astype('datetime64[us]')
        stft_mpl = mdates.date2num(stft_dts)

        fig, ax = plt.subplots(figsize=(20, 6), facecolor=STYLE['bg'])
        ax.set_facecolor(STYLE['bg'])
        freq_mask = freqs <= FREQ_MAX_HZ
        f_sub = freqs[freq_mask]

        im = ax.pcolormesh(
            stft_mpl, f_sub, spec_db[freq_mask, :],
            cmap=STYLE['cmap'], vmin=-3, vmax=20, shading='auto', rasterized=True
        )
        ax.xaxis_date()
        ax.xaxis.set_major_formatter(_DF('%H:%M'))
        ax.xaxis.set_major_locator(_ML(10))
        cbar = plt.colorbar(im, ax=ax, pad=0.01, shrink=0.9)
        cbar.set_label('dB above background', color=STYLE['text'], fontsize=10)
        cbar.ax.tick_params(colors=STYLE['text'])
        ax.set_xlabel('Time (IST)', color=STYLE['text'], fontsize=11)
        ax.set_ylabel('Frequency (Hz)', color=STYLE['text'], fontsize=11)
        ax.set_title(f'LOFAR — session_{session} — {file_start.strftime("%d %b %Y %H:%M")} IST — {sr} Hz',
                      color=STYLE['text'], fontsize=13, fontweight='bold', pad=10)
        ax.tick_params(colors=STYLE['text'])
        for s in ax.spines.values():
            s.set_color(STYLE['box_edge'])

        img_path = os.path.join(OUTPUT_DIR, f'lofar_session_{session}.png')
        plt.savefig(img_path, dpi=120, facecolor=STYLE['bg'], bbox_inches='tight')
        plt.close()

    # ---- Process 30-second analysis windows ----
    results = []
    window_samples = int(ANALYSIS_WINDOW_SEC * sr)
    hop_samples = int(ANALYSIS_HOP_SEC * sr)
    n_windows = max(1, (len(audio) - window_samples) // hop_samples + 1)

    demon_count = 0
    for wi in range(n_windows):
        start_sample = wi * hop_samples
        end_sample = start_sample + window_samples
        if end_sample > len(audio):
            break

        chunk = audio[start_sample:end_sample]
        chunk_start_sec = start_sample / sr

        # Find STFT columns that fall within this window
        stft_mask = (stft_times >= chunk_start_sec) & (stft_times < end_sample / sr)
        if not stft_mask.any():
            continue
        spec_chunk = spec_db[:, stft_mask]

        # Timestamp for this window
        ts = file_start + timedelta(seconds=chunk_start_sec)

        # Process window (Stage 3 runs if rms_mean_global provided)
        features = process_analysis_window(
            chunk, sr, spec_chunk, freqs, stft_times[stft_mask],
            rms_mean_global=rms_mean_global
        )

        if features['demon_confidence'] != 'NONE':
            demon_count += 1

        # Add metadata
        features['file'] = fname
        features['session'] = session
        features['window_idx'] = wi
        features['time_offset_sec'] = round(chunk_start_sec, 2)
        features['timestamp'] = ts.isoformat()
        features['sample_rate'] = sr

        results.append(features)

    # Persistent tonals for the full file
    persistent = detect_persistent_tonals(spec_db, freqs, stft_times)
    n_persistent = len(persistent)
    if persistent:
        top_freqs = ', '.join(f'{t["freq_hz"]:.1f}Hz ({t["duration_sec"]:.0f}s)' for t in persistent[:5])
    else:
        top_freqs = 'none'
    demon_msg = f', DEMON on {demon_count} windows' if demon_count > 0 else ''
    print(f'    {n_windows} windows, {n_persistent} persistent tonals{demon_msg}: {top_freqs}')

    # Clean up
    del audio, Sxx, spec_db
    gc.collect()

    return results


# ---- Main processing loop ----
# Pass 1: Stages 1+2 to establish global RMS baseline
# Pass 2: Full pipeline with DEMON gating

processed_sessions = set()
all_results = []

# Check for checkpoint
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    processed_sessions = set(ckpt.get('processed_sessions', []))
    csv_path = os.path.join(OUTPUT_DIR, 'lofar_hydrophone_results.csv')
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        # Check if DEMON columns exist (i.e., Pass 2 already done)
        if 'blade_rate_hz' in existing.columns:
            all_results = existing.to_dict('records')
            print(f'Resuming from Pass 2: {len(processed_sessions)} files, {len(all_results)} windows')
        else:
            all_results = existing.to_dict('records')
            print(f'Resuming from Pass 1: {len(processed_sessions)} files, {len(all_results)} windows')
else:
    print('Starting fresh — Pass 1 (Stages 1+2)')

start_time = _time.time()

# ---- PASS 1: Stages 1+2 (if not already done) ----
if not all_results or 'blade_rate_hz' not in all_results[0]:
    files_to_process = [f for f in all_files if f['session'] not in processed_sessions]
    print(f'Pass 1: {len(files_to_process)} files to process')
    print()

    for fi, file_info in enumerate(files_to_process):
        file_results = process_one_file(file_info, fi, len(files_to_process),
                                         rms_mean_global=None)
        all_results.extend(file_results)
        processed_sessions.add(file_info['session'])

        if (fi + 1) % 5 == 0 or fi == len(files_to_process) - 1:
            df_out = pd.DataFrame(all_results)
            csv_path = os.path.join(OUTPUT_DIR, 'lofar_hydrophone_results.csv')
            df_out.to_csv(csv_path, index=False)
            with open(CHECKPOINT_FILE, 'w') as f:
                json.dump({'processed_sessions': sorted(processed_sessions)}, f)
            elapsed = _time.time() - start_time
            rate = (fi + 1) / (elapsed / 60) if elapsed > 0 else 0
            remaining = (len(files_to_process) - fi - 1) / rate if rate > 0 else 0
            print(f'    Checkpoint: {len(processed_sessions)} files, {len(all_results)} windows, '
                  f'{elapsed/60:.1f}min elapsed, ~{remaining:.0f}min remaining')

    pass1_time = _time.time() - start_time
    print(f'\nPass 1 complete: {pass1_time/60:.1f} min')

    # Compute global RMS mean from Pass 1
    df_pass1 = pd.DataFrame(all_results)
    rms_mean_global = df_pass1['rms_energy'].mean()
    print(f'Global RMS mean: {rms_mean_global:.6f}')
    print(f'DEMON gate threshold: {DEMON_RMS_GATE}× = {rms_mean_global * DEMON_RMS_GATE:.6f}')
    n_eligible = ((df_pass1['rms_energy'] >= rms_mean_global * DEMON_RMS_GATE) &
                  (df_pass1['n_tonals'] >= DEMON_MIN_TONALS)).sum()
    print(f'Windows eligible for DEMON: {n_eligible} ({100*n_eligible/len(df_pass1):.1f}%)')

    # ---- PASS 2: Re-process ALL files with DEMON enabled ----
    print(f'\nPass 2: Full pipeline (Stages 1+2+3) — {len(all_files)} files')
    print()
    all_results = []
    processed_sessions = set()
    pass2_start = _time.time()

    for fi, file_info in enumerate(all_files):
        file_results = process_one_file(file_info, fi, len(all_files),
                                         rms_mean_global=rms_mean_global)
        all_results.extend(file_results)
        processed_sessions.add(file_info['session'])

        if (fi + 1) % 5 == 0 or fi == len(all_files) - 1:
            df_out = pd.DataFrame(all_results)
            csv_path = os.path.join(OUTPUT_DIR, 'lofar_hydrophone_results.csv')
            df_out.to_csv(csv_path, index=False)
            with open(CHECKPOINT_FILE, 'w') as f:
                json.dump({'processed_sessions': sorted(processed_sessions),
                           'pass2_complete': fi == len(all_files) - 1}, f)
            elapsed = _time.time() - pass2_start
            rate = (fi + 1) / (elapsed / 60) if elapsed > 0 else 0
            remaining = (len(all_files) - fi - 1) / rate if rate > 0 else 0
            print(f'    Checkpoint: {len(processed_sessions)} files, {len(all_results)} windows, '
                  f'{elapsed/60:.1f}min elapsed, ~{remaining:.0f}min remaining')

    pass2_time = _time.time() - pass2_start
    print(f'\nPass 2 complete: {pass2_time/60:.1f} min')

total_time = _time.time() - start_time
print(f'\n{"="*60}')
print(f'PROCESSING COMPLETE')
print(f'  Files: {len(processed_sessions)}')
print(f'  Windows: {len(all_results)}')
print(f'  Total time: {total_time/60:.1f} minutes')
print(f'  Output: {OUTPUT_DIR}')


Starting fresh — Pass 1 (Stages 1+2)
Pass 1: 4 files to process

  [1/4] session_28jan_1 — 28 Jan 13:25 (Stages 1+2)
    512 Hz, 3595s, 1.8M samples
    238 windows, 661 persistent tonals: 30.0Hz (268s), 35.2Hz (107s), 35.5Hz (122s), 26.4Hz (159s), 58.0Hz (374s)
  [2/4] session_28jan_2 — 28 Jan 14:25 (Stages 1+2)
    512 Hz, 3595s, 1.8M samples
    238 windows, 269 persistent tonals: 37.0Hz (105s), 37.9Hz (60s), 30.5Hz (31s), 60.5Hz (249s), 37.0Hz (58s)
  [3/4] session_28jan_3 — 28 Jan 15:25 (Stages 1+2)
    512 Hz, 3595s, 1.8M samples
    238 windows, 126 persistent tonals: 134.4Hz (66s), 134.4Hz (51s), 134.4Hz (115s), 18.9Hz (182s), 134.4Hz (49s)
  [4/4] session_28jan_4 — 28 Jan 16:25 (Stages 1+2)
    512 Hz, 1153s, 0.6M samples
    75 windows, 97 persistent tonals: 56.2Hz (181s), 51.4Hz (172s), 62.4Hz (180s), 47.6Hz (171s), 61.1Hz (317s)
    Checkpoint: 4 files, 789 windows, 0.3min elapsed, ~0min remaining

Pass 1 complete: 0.3 min
Global RMS mean: 0.000479
DEMON gate threshold: 3.0

## 7. Analysis & Event Detection

Reads the per-window CSV and finds vessel transit events using RMS energy
combined with tonal detection. Also summarises DEMON results.


In [ ]:
# Load results
csv_path = os.path.join(OUTPUT_DIR, 'lofar_hydrophone_results.csv')
df = pd.read_csv(csv_path)
df['ts'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('ts').reset_index(drop=True)

print(f'Loaded {len(df)} windows')
print(f'Date range: {df["ts"].min()} → {df["ts"].max()}')

# ---- Summary statistics ----
rms_mean = df['rms_energy'].mean()
rms_std = df['rms_energy'].std()
rms_thresh = rms_mean + RMS_SIGMA_THRESHOLD * rms_std

print(f'\n--- STAGE 1: RMS ENERGY ---')
print(f'  Range: {df["rms_energy"].min():.6f} – {df["rms_energy"].max():.6f}')
print(f'  Dynamic range: {df["rms_energy"].max()/df["rms_energy"].min():.0f}×')
print(f'  Mean: {rms_mean:.6f}, Std: {rms_std:.6f}')
print(f'  Detection threshold ({RMS_SIGMA_THRESHOLD}σ): {rms_thresh:.6f}')
for sigma in [1.0, 1.5, 2.0, 3.0]:
    t = rms_mean + sigma * rms_std
    n = (df['rms_energy'] > t).sum()
    print(f'    >{sigma}σ: {n} windows ({100*n/len(df):.1f}%)')

print(f'\n--- STAGE 2: TONAL DETECTION ---')
print(f'  Windows with ≥1 tonal: {(df["n_tonals"] > 0).sum()} ({100*(df["n_tonals"] > 0).sum()/len(df):.1f}%)')
print(f'  Windows with ≥3 tonals: {(df["n_tonals"] >= 3).sum()} ({100*(df["n_tonals"] >= 3).sum()/len(df):.1f}%)')
print(f'  Max tonals in one window: {df["n_tonals"].max()}')
print(f'  Mean tonal energy: {df["tonal_energy"].mean():.2f}')
print(f'  Max tonal energy: {df["tonal_energy"].max():.2f}')

print(f'\n--- STAGE 3: DEMON (conditional) ---')
demon_attempted = df['demon_confidence'] != 'NONE'
demon_high = df['demon_confidence'] == 'HIGH'
demon_low = df['demon_confidence'] == 'LOW'
print(f'  DEMON attempted: {demon_attempted.sum()} windows ({100*demon_attempted.sum()/len(df):.1f}%)')
print(f'  HIGH confidence: {demon_high.sum()} windows')
print(f'  LOW confidence:  {demon_low.sum()} windows')
print(f'  Not attempted:   {(~demon_attempted).sum()} windows')
if demon_attempted.any():
    demon_df = df[demon_attempted]
    print(f'  Blade rate range: {demon_df["blade_rate_hz"].min():.1f} – {demon_df["blade_rate_hz"].max():.1f} Hz')
    print(f'  Shaft rate range: {demon_df["shaft_rate_hz"].min():.2f} – {demon_df["shaft_rate_hz"].max():.2f} Hz')
    if demon_high.any():
        high_df = df[demon_high]
        print(f'  HIGH-confidence blade rates: {high_df["blade_rate_hz"].median():.1f} Hz (median)')

# ---- Soft-gated vessel score ----
rms_ratio = df['rms_energy'] / rms_mean
gate_weight = np.clip(rms_ratio / 2.0, 0.0, 1.0)
gated_n_tonals = df['n_tonals'] * gate_weight
gated_tonal_energy = df['tonal_energy'] * gate_weight

df['vessel_score'] = (
    np.log10(1 + rms_ratio) *
    (1 + gated_n_tonals) *
    (1 + np.log10(1 + gated_tonal_energy))
)

save_cols = [c for c in df.columns if c not in ('ts',)]
df[save_cols].to_csv(csv_path, index=False)
print(f'\nUpdated CSV saved with soft-gated vessel_score')

print(f'\n--- VESSEL SCORE ---')
print(f'  Range: {df["vessel_score"].min():.2f} – {df["vessel_score"].max():.2f}')

day = df[(df['ts'].dt.hour >= 6) & (df['ts'].dt.hour < 18)]
night = df[(df['ts'].dt.hour >= 18) | (df['ts'].dt.hour < 6)]
print(f'\n--- DAY / NIGHT ---')
print(f'  Day mean RMS:     {day["rms_energy"].mean():.6f}')
print(f'  Night mean RMS:   {night["rms_energy"].mean():.6f}')
if night['rms_energy'].mean() > 0:
    print(f'  RMS ratio:        {day["rms_energy"].mean()/night["rms_energy"].mean():.2f}×')

# ---- Event detection ----
df['elevated'] = df['rms_energy'] > rms_thresh
elevated_idx = set(df[df['elevated']].index)
expanded = set()
for idx in elevated_idx:
    for offset in range(-EVENT_MERGE_RADIUS, EVENT_MERGE_RADIUS + 1):
        if 0 <= idx + offset < len(df):
            expanded.add(idx + offset)

df['in_event'] = df.index.isin(expanded)
df['event_group'] = (df['in_event'] != df['in_event'].shift()).cumsum()

events = []
for gid, group in df[df['in_event']].groupby('event_group'):
    if len(group) < EVENT_MIN_WINDOWS:
        continue
    start = group['ts'].iloc[0]
    end = group['ts'].iloc[-1]
    dur_min = (end - start).total_seconds() / 60
    peak_rms = group['rms_energy'].max()

    # DEMON summary for this event
    demon_windows = group[group['demon_confidence'] != 'NONE']
    if len(demon_windows) > 0:
        ev_blade_rate = demon_windows['blade_rate_hz'].median()
        ev_shaft_rate = demon_windows['shaft_rate_hz'].median()
        ev_n_blades = int(demon_windows['est_n_blades'].mode().iloc[0]) if len(demon_windows) > 0 else None
        ev_rpm = demon_windows['est_rpm'].median()
        ev_demon_conf = 'HIGH' if (demon_windows['demon_confidence'] == 'HIGH').any() else 'LOW'
    else:
        ev_blade_rate = None
        ev_shaft_rate = None
        ev_n_blades = None
        ev_rpm = None
        ev_demon_conf = 'NONE'

    events.append({
        'event_id': len(events) + 1,
        'start': start, 'end': end, 'dur_min': dur_min,
        'n_windows': len(group),
        'peak_rms': peak_rms,
        'mean_rms': group['rms_energy'].mean(),
        'rms_ratio': peak_rms / rms_mean,
        'mean_tonals': group['n_tonals'].mean(),
        'max_tonals': group['n_tonals'].max(),
        'mean_tonal_energy': group['tonal_energy'].mean(),
        'mean_vessel_score': group['vessel_score'].mean(),
        'max_vessel_score': group['vessel_score'].max(),
        'strongest_tonal_hz': group.loc[group['strongest_tonal_db'].idxmax(), 'strongest_tonal_hz'] if group['strongest_tonal_db'].max() > 0 else 0,
        'blade_rate_hz': ev_blade_rate,
        'shaft_rate_hz': ev_shaft_rate,
        'est_n_blades': ev_n_blades,
        'est_rpm': ev_rpm,
        'demon_confidence': ev_demon_conf,
        'files': sorted(group['file'].unique().tolist()),
    })

# Classify events
significant = []
for ev in events:
    is_sig = ev['dur_min'] >= SIGNIFICANT_DURATION_MIN or ev['rms_ratio'] >= SIGNIFICANT_RMS_FACTOR
    has_tonals = ev['mean_vessel_score'] >= 2.0
    if is_sig:
        if has_tonals and ev['dur_min'] >= 8:
            ev['assessment'] = 'CONFIRMED VESSEL'
        elif has_tonals:
            ev['assessment'] = 'PROBABLE VESSEL'
        elif ev['rms_ratio'] >= 10:
            ev['assessment'] = 'HIGH ENERGY (no tonals)'
        else:
            ev['assessment'] = 'ELEVATED ACTIVITY'
        significant.append(ev)
    else:
        ev['assessment'] = 'brief transient'

print(f'\n{"="*60}')
print(f'EVENT DETECTION RESULTS')
print(f'{"="*60}')
print(f'  Total events: {len(events)}')
print(f'  Significant: {len(significant)}')
print(f'\n{"#":>3} {"Time":<26} {"Dur":>5} {"Peak":>7} {"Tonals":>7} {"Score":>6} {"BR":>6} {"SR":>6} {"Conf":>5} {"Assessment"}')
print("-"*105)
for ev in sorted(significant, key=lambda e: e['rms_ratio'], reverse=True):
    br_str = f'{ev["blade_rate_hz"]:.1f}' if ev['blade_rate_hz'] else '  —'
    sr_str = f'{ev["shaft_rate_hz"]:.2f}' if ev['shaft_rate_hz'] else '  —'
    conf_str = ev['demon_confidence'][:3] if ev['demon_confidence'] != 'NONE' else ' —'
    print(f'{ev["event_id"]:3d} {ev["start"].strftime("%d %b %H:%M")}–{ev["end"].strftime("%H:%M")}'
          f'  {ev["dur_min"]:4.0f}m {ev["rms_ratio"]:6.1f}× '
          f'{ev["mean_tonals"]:6.1f} {ev["max_vessel_score"]:5.1f} '
          f'{br_str:>6} {sr_str:>6} {conf_str:>5}  '
          f'{ev["assessment"]}')

# Save events CSV
events_df = pd.DataFrame([{
    'event_id': e['event_id'],
    'start': e['start'].isoformat(),
    'end': e['end'].isoformat(),
    'dur_min': round(e['dur_min'], 1),
    'n_windows': e['n_windows'],
    'peak_rms': round(e['peak_rms'], 6),
    'rms_ratio': round(e['rms_ratio'], 1),
    'mean_tonals': round(e['mean_tonals'], 2),
    'max_tonals': e['max_tonals'],
    'mean_tonal_energy': round(e['mean_tonal_energy'], 2),
    'mean_vessel_score': round(e['mean_vessel_score'], 2),
    'max_vessel_score': round(e['max_vessel_score'], 2),
    'strongest_tonal_hz': round(e['strongest_tonal_hz'], 2),
    'blade_rate_hz': round(e['blade_rate_hz'], 2) if e['blade_rate_hz'] else None,
    'shaft_rate_hz': round(e['shaft_rate_hz'], 3) if e['shaft_rate_hz'] else None,
    'est_n_blades': e['est_n_blades'],
    'est_rpm': round(e['est_rpm'], 1) if e['est_rpm'] else None,
    'demon_confidence': e['demon_confidence'],
    'assessment': e['assessment'],
} for e in events])
events_path = os.path.join(OUTPUT_DIR, 'transit_events.csv')
events_df.to_csv(events_path, index=False)
print(f'\nEvents saved to: {events_path}')


Loaded 789 windows
Date range: 2026-01-28 13:25:19 → 2026-01-28 16:43:48

--- STAGE 1: RMS ENERGY ---
  Range: 0.000199 – 0.004172
  Dynamic range: 21×
  Mean: 0.000479, Std: 0.000362
  Detection threshold (1.0σ): 0.000842
    >1.0σ: 63 windows (8.0%)
    >1.5σ: 38 windows (4.8%)
    >2.0σ: 20 windows (2.5%)
    >3.0σ: 12 windows (1.5%)

--- STAGE 2: TONAL DETECTION ---
  Windows with ≥1 tonal: 517 (65.5%)
  Windows with ≥3 tonals: 336 (42.6%)
  Max tonals in one window: 82
  Mean tonal energy: 73.52
  Max tonal energy: 1258.04

--- STAGE 3: DEMON (conditional) ---
  DEMON attempted: 13 windows (1.6%)
  HIGH confidence: 7 windows
  LOW confidence:  6 windows
  Not attempted:   776 windows
  Blade rate range: 2.2 – 3.4 Hz
  Shaft rate range: 0.54 – 0.82 Hz
  HIGH-confidence blade rates: 2.7 Hz (median)

Updated CSV saved with soft-gated vessel_score

--- VESSEL SCORE ---
  Range: 0.15 – 315.72

--- DAY / NIGHT ---
  Day mean RMS:     0.000479
  Night mean RMS:   nan

EVENT DETECTION RES

## 8. Generate Event Spectrograms

Zoomed LOFAR spectrograms for the largest events — 3-panel figures showing
the normalised spectrogram, RMS energy profile, and detected tonals.


In [ ]:
if not GENERATE_EVENT_SPECTROGRAMS:
    print('Event spectrogram generation disabled')
else:
    import matplotlib.dates as mdates
    from matplotlib.dates import DateFormatter as _DF, AutoDateLocator as _ADL
    import matplotlib.gridspec as _gs
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    large_events = [e for e in events if e['rms_ratio'] >= EVENT_SPEC_MIN_RMS_FACTOR]
    large_events = sorted(large_events, key=lambda e: e['rms_ratio'], reverse=True)
    print(f'Generating spectrograms for {len(large_events)} large events (≥{EVENT_SPEC_MIN_RMS_FACTOR}× background)')

    event_spec_start = _time.time()

    for eidx, ev in enumerate(large_events):
        print(f'  [{eidx+1}/{len(large_events)}] Event {ev["event_id"]}: '
              f'{ev["start"].strftime("%d %b %H:%M")}–{ev["end"].strftime("%H:%M")} '
              f'({ev["rms_ratio"]:.0f}×)')

        padded_start = ev['start'] - timedelta(minutes=EVENT_PADDING_MIN)
        padded_end = ev['end'] + timedelta(minutes=EVENT_PADDING_MIN)

        candidate_files = []
        for finfo in all_files:
            if abs((finfo['start_ts'] - ev['start']).total_seconds()) < 7200:
                candidate_files.append(finfo)
        candidate_files = sorted(candidate_files, key=lambda f: f['start_ts'])

        if not candidate_files:
            print('    SKIP: no WAV files found')
            continue

        all_audio = []
        earliest_ts = None
        target_sr = None

        for finfo in candidate_files:
            try:
                data, sr = sf.read(finfo['path'])
            except Exception as e:
                print(f'    WARN: could not read {finfo["path"]}: {e}')
                continue
            if data.ndim > 1:
                data = data[:, 0]
            file_end = finfo['start_ts'] + timedelta(seconds=len(data) / sr)

            if file_end < padded_start or finfo['start_ts'] > padded_end:
                continue

            if earliest_ts is None:
                earliest_ts = finfo['start_ts']
                target_sr = sr

            all_audio.append({
                'data': data, 'sr': sr, 'start': finfo['start_ts']
            })

        if not all_audio or earliest_ts is None:
            print('    SKIP: no overlapping audio')
            continue

        combined = np.concatenate([a['data'] for a in sorted(all_audio, key=lambda a: a['start'])])
        sr = target_sr

        start_offset = max(0, int((padded_start - earliest_ts).total_seconds() * sr))
        end_offset = min(len(combined), int((padded_end - earliest_ts).total_seconds() * sr))
        audio = combined[start_offset:end_offset]
        audio_start_ts = earliest_ts + timedelta(seconds=start_offset / sr)

        if len(audio) < sr * 10:
            print('    SKIP: audio too short')
            continue

        freqs, stft_times, Sxx = compute_stft(audio, sr)
        spec_db, _ = normalise_spectrogram(freqs, stft_times, Sxx)

        stft_mpl = mdates.date2num(np.array(
            [audio_start_ts + timedelta(seconds=float(t)) for t in stft_times],
            dtype='datetime64[ms]'
        ).astype('datetime64[us]'))

        persistent = detect_persistent_tonals(spec_db, freqs, stft_times)
        n_persistent = len(persistent)
        for t in persistent:
            t['start_dt'] = mdates.date2num(audio_start_ts + timedelta(seconds=t['start_sec']))
            t['end_dt'] = mdates.date2num(audio_start_ts + timedelta(seconds=t['end_sec']))

        rms_win = int(sr * 1.0)
        rms_mpl = []
        rms_v = []
        for t_samp in range(0, len(audio) - rms_win, rms_win):
            rms_v.append(float(np.sqrt(np.mean(audio[t_samp:t_samp + rms_win] ** 2))))
            rms_mpl.append(mdates.date2num(audio_start_ts + timedelta(seconds=t_samp / sr)))

        ev_start_mpl = mdates.date2num(ev['start'])
        ev_end_mpl = mdates.date2num(ev['end'])

        total_dur_sec = (padded_end - padded_start).total_seconds()
        tick_interval_min = max(1, round(total_dur_sec / 60 / 5))

        fig = plt.figure(figsize=(20, 13), facecolor=STYLE['bg'])
        gs = _gs.GridSpec(3, 1, height_ratios=[5, 1.6, 1.2], hspace=0.05, figure=fig,
                          left=0.06, right=0.94, top=0.93, bottom=0.06)

        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1], sharex=ax1)
        ax3 = fig.add_subplot(gs[2], sharex=ax1)

        ax1.set_facecolor(STYLE['bg'])
        freq_mask = freqs <= FREQ_MAX_HZ
        f_sub = freqs[freq_mask]
        im = ax1.pcolormesh(
            stft_mpl, f_sub, spec_db[freq_mask, :],
            cmap=STYLE['cmap'], vmin=-3, vmax=20, shading='auto', rasterized=True
        )
        ax1.axvline(ev_start_mpl, color='#00ff88', lw=1.5, ls='--', alpha=0.8)
        ax1.axvline(ev_end_mpl, color='#00ff88', lw=1.5, ls='--', alpha=0.8)

        for t in persistent[:30]:
            ax1.plot([t['start_dt'], t['end_dt']], [t['freq_hz'], t['freq_hz']],
                     color='#00ff88', lw=1.0, alpha=0.4)

        cbax = ax1.inset_axes([1.01, 0.0, 0.015, 1.0])
        cbar = fig.colorbar(im, cax=cbax)
        cbar.set_label('dB above background', color=STYLE['text'], fontsize=10)
        cbar.ax.tick_params(colors=STYLE['text'])

        ax1.set_ylabel('Frequency (Hz)', color=STYLE['text'], fontsize=11)
        ax1.tick_params(colors=STYLE['text'])
        plt.setp(ax1.get_xticklabels(), visible=False)
        for s in ax1.spines.values():
            s.set_color(STYLE['box_edge'])

        # Title includes DEMON info if available
        demon_str = ''
        if ev.get('blade_rate_hz') and ev.get('demon_confidence') != 'NONE':
            demon_str = f' — BR {ev["blade_rate_hz"]:.1f} Hz, {ev["est_n_blades"]}-blade {ev["est_rpm"]:.0f} RPM ({ev["demon_confidence"]})'
        title = (f'LOFAR — Event {ev["event_id"]} — '
                 f'{ev["start"].strftime("%d %b %H:%M")}–{ev["end"].strftime("%H:%M")} IST — '
                 f'{ev["rms_ratio"]:.0f}× peak — '
                 f'{n_persistent} tonals — {ev["assessment"]}{demon_str}')
        ax1.set_title(title, color=STYLE['text'], fontsize=12, fontweight='bold', pad=12)

        ax2.set_facecolor(STYLE['bg'])
        ax2.fill_between(rms_mpl, rms_v, color=STYLE['accent'], alpha=0.5)
        ax2.plot(rms_mpl, rms_v, color=STYLE['accent'], lw=0.8)
        ax2.axvline(ev_start_mpl, color='#00ff88', lw=1.5, ls='--', alpha=0.8)
        ax2.axvline(ev_end_mpl, color='#00ff88', lw=1.5, ls='--', alpha=0.8)
        ax2.set_ylabel('RMS Energy', color=STYLE['text'], fontsize=10)
        ax2.tick_params(colors=STYLE['text'])
        plt.setp(ax2.get_xticklabels(), visible=False)
        ax2.grid(True, alpha=0.15, color=STYLE['box_edge'])
        for s in ax2.spines.values():
            s.set_color(STYLE['box_edge'])

        ax3.set_facecolor(STYLE['bg'])
        for t in persistent:
            ax3.plot([t['start_dt'], t['end_dt']], [t['freq_hz'], t['freq_hz']],
                     color=STYLE['highlight'], lw=2, alpha=0.7, solid_capstyle='round')
        ax3.axvline(ev_start_mpl, color='#00ff88', lw=1.5, ls='--', alpha=0.8)
        ax3.axvline(ev_end_mpl, color='#00ff88', lw=1.5, ls='--', alpha=0.8)
        ax3.set_ylabel('Tonal (Hz)', color=STYLE['text'], fontsize=10)
        ax3.set_ylim(0, FREQ_MAX_HZ)
        ax3.tick_params(colors=STYLE['text'])
        ax3.grid(True, alpha=0.15, color=STYLE['box_edge'])
        for s in ax3.spines.values():
            s.set_color(STYLE['box_edge'])

        from matplotlib.dates import MinuteLocator as _ML
        ax3.xaxis.set_major_formatter(_DF('%H:%M'))
        ax3.xaxis.set_major_locator(_ML(tick_interval_min))
        ax3.set_xlabel('Time (IST)', color=STYLE['text'], fontsize=11)

        safe_name = ev['start'].strftime('%Y%m%d_%H%M')
        outpath = os.path.join(OUTPUT_DIR,
            f'lofar_event_{ev["event_id"]:02d}_{safe_name}_{ev["rms_ratio"]:.0f}x.png')
        plt.savefig(outpath, dpi=150, facecolor=STYLE['bg'], bbox_inches='tight')
        plt.close()
        print(f'    Saved: {os.path.basename(outpath)}')

        del audio, combined, Sxx, spec_db
        gc.collect()

    print(f'\nEvent spectrograms complete: {(_time.time() - event_spec_start)/60:.1f} min')


Generating spectrograms for 0 large events (≥10.0× background)

Event spectrograms complete: 0.0 min


## 9. 48-Hour Timeline Visualisation

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(22, 20), sharex=True, facecolor=STYLE['bg'])
for ax in axes:
    ax.set_facecolor(STYLE['bg'])
    ax.tick_params(colors=STYLE['text'], labelsize=9)
    for s in ax.spines.values():
        s.set_color(STYLE['box_edge'])
    ax.grid(True, alpha=0.2, color=STYLE['box_edge'])

from matplotlib.dates import DateFormatter

# Panel 1: RMS Energy
ax = axes[0]
colors = [STYLE['red'] if r > rms_thresh else '#2c3e50' for r in df['rms_energy']]
ax.scatter(df['ts'], df['rms_energy'], c=colors, s=3, alpha=0.5)
ax.axhline(rms_thresh, color=STYLE['accent'], ls='--', lw=1,
           label=f'{RMS_SIGMA_THRESHOLD}σ threshold')
ax.set_ylabel('RMS Energy', color=STYLE['text'], fontsize=11)
ax.set_yscale('log')
ax.legend(loc='upper right', fontsize=9, facecolor=STYLE['bg'],
          edgecolor=STYLE['box_edge'], labelcolor=STYLE['text'])
date_start = df['ts'].iloc[0].strftime('%d %b')
date_end = df['ts'].iloc[-1].strftime('%d %b %Y')
ax.set_title(f'LOFAR + DEMON Hydrophone Analysis — NUWR Goa — {date_start} – {date_end}',
             color=STYLE['text'], fontsize=14, fontweight='bold', pad=12)

# Panel 2: Vessel Score
ax = axes[1]
ax.scatter(df['ts'], df['vessel_score'], c=STYLE['accent'], s=3, alpha=0.4)
ax.axhline(2.0, color=STYLE['highlight'], ls='--', lw=1, label='Score = 2')
ax.set_ylabel('Vessel Score', color=STYLE['text'], fontsize=11)
ax.legend(loc='upper right', fontsize=9, facecolor=STYLE['bg'],
          edgecolor=STYLE['box_edge'], labelcolor=STYLE['text'])

# Panel 3: Number of Tonals
ax = axes[2]
ax.scatter(df['ts'], df['n_tonals'], c=STYLE['green'], s=3, alpha=0.4)
ax.set_ylabel('Tonals Detected', color=STYLE['text'], fontsize=11)

# Panel 4: Strongest Tonal Frequency
ax = axes[3]
has_tonal = df['strongest_tonal_hz'] > 0
ax.scatter(df['ts'][has_tonal], df['strongest_tonal_hz'][has_tonal],
           c=STYLE['highlight'], s=5, alpha=0.4)
ax.set_ylabel('Strongest Tonal (Hz)', color=STYLE['text'], fontsize=11)
ax.set_ylim(0, 260)

# Panel 5: DEMON Blade Rate (only where attempted)
ax = axes[4]
demon_mask = df['demon_confidence'] != 'NONE'
high_mask = df['demon_confidence'] == 'HIGH'
low_mask = df['demon_confidence'] == 'LOW'
if low_mask.any():
    ax.scatter(df['ts'][low_mask], df['blade_rate_hz'][low_mask],
               c=STYLE['accent'], s=8, alpha=0.4, label='LOW confidence')
if high_mask.any():
    ax.scatter(df['ts'][high_mask], df['blade_rate_hz'][high_mask],
               c=STYLE['green'], s=15, alpha=0.8, label='HIGH confidence', zorder=5)
ax.set_ylabel('Blade Rate (Hz)', color=STYLE['text'], fontsize=11)
ax.set_ylim(0, 30)
ax.legend(loc='upper right', fontsize=9, facecolor=STYLE['bg'],
          edgecolor=STYLE['box_edge'], labelcolor=STYLE['text'])
ax.xaxis.set_major_formatter(DateFormatter('%d %b\n%H:%M'))
ax.set_xlabel('Time (IST)', color=STYLE['text'], fontsize=11)

plt.tight_layout()
timeline_path = os.path.join(OUTPUT_DIR, 'timeline_48hr_lofar_demon.png')
plt.savefig(timeline_path, dpi=150, facecolor=STYLE['bg'], bbox_inches='tight')
plt.close()
print(f'Timeline saved to: {timeline_path}')


Timeline saved to: /content/drive/MyDrive/SKANN_SSL/lofar_results_28jan/timeline_48hr_lofar_demon.png


## 10. Customer-Format Output (XLSX)

Generates the spreadsheet in Altair's presentation format:
- Multi-row per event (one row per primary frequency)
- Harmonics checked against detected tonal set (shows "2xf, 3xf" or "Not Present")
- Bearing columns left blank (single hydrophone)
- DEMON blade rate / shaft rate with confidence in Remarks


In [ ]:
# Generate customer-format XLSX from events
# Multi-row per event matching Altair presentation format

!pip install openpyxl -q

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
ws = wb.active
ws.title = 'HAVS Results'

# Styles
header_font_white = Font(bold=True, size=10, name='Arial', color='FFFFFF')
header_fill = PatternFill('solid', fgColor='4472C4')
sub_header_fill = PatternFill('solid', fgColor='D6E4F0')
sub_header_font = Font(bold=True, size=9, name='Arial')
data_font = Font(size=9, name='Arial')
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)
center = Alignment(horizontal='center', vertical='center', wrap_text=True)
left_align = Alignment(horizontal='left', vertical='center', wrap_text=True)

# Row 1: Main headers (merged)
main_headers = {
    'A1': 'Date', 'B1': 'Time',
    'C1': 'Spectrogram Features',
    'E1': 'Bearing Features',
    'H1': 'LOFAR Features',
    'J1': 'DEMON Features',
    'L1': 'Event', 'M1': 'Remarks'
}
for cell_ref, text in main_headers.items():
    ws[cell_ref] = text
    ws[cell_ref].font = header_font_white
    ws[cell_ref].fill = header_fill
    ws[cell_ref].alignment = center
    ws[cell_ref].border = thin_border

ws.merge_cells('C1:D1')
ws.merge_cells('E1:G1')
ws.merge_cells('H1:I1')
ws.merge_cells('J1:K1')

for col in range(1, 14):
    cell = ws.cell(row=1, column=col)
    cell.border = thin_border
    if not cell.value:
        cell.fill = header_fill

# Row 2: Sub-headers
sub_headers = [
    '', '',
    'Primary Frequencies (Hz)', 'Harmonics (Hz)',
    'Bearing Angle (\u00b0)', 'BTR Curve Feature', 'Estimated Speed (kn)',
    'Primary Frequencies (Hz)', 'Harmonics (Hz)',
    'Blade Rate (Hz)', 'Shaft Rate (Hz)',
    '', ''
]
for col, text in enumerate(sub_headers, 1):
    cell = ws.cell(row=2, column=col, value=text)
    cell.font = sub_header_font
    cell.fill = sub_header_fill
    cell.alignment = center
    cell.border = thin_border

# Column widths
for col_letter, w in {'A': 12, 'B': 14, 'C': 18, 'D': 20, 'E': 12, 'F': 14,
                       'G': 14, 'H': 18, 'I': 20, 'J': 12, 'K': 12, 'L': 18, 'M': 40}.items():
    ws.column_dimensions[col_letter].width = w

# Process events into multi-row format
row = 3
current_date = None

for ev in sorted(events, key=lambda e: e['start']):
    if ev['assessment'] == 'brief transient':
        continue

    date_str = ev['start'].strftime('%d/%m/%Y')
    time_str = f'{ev["start"].strftime("%H:%M")}-{ev["end"].strftime("%H:%M")}'

    # Get all windows for this event
    mask = (df['ts'] >= ev['start']) & (df['ts'] <= ev['end'])
    ev_windows = df[mask]

    # Collect all tonal frequencies from this event
    all_tonals = []
    for _, win_row in ev_windows.iterrows():
        tf = str(win_row.get('tonal_freqs', ''))
        if tf and tf != '' and tf != 'nan':
            for f in tf.split('|'):
                try:
                    all_tonals.append(float(f))
                except ValueError:
                    pass

    # Get top primary frequencies by occurrence
    if all_tonals:
        from collections import Counter
        rounded = [round(f, 0) for f in all_tonals]
        freq_counts = Counter(rounded)
        primary_freqs = [f for f, _ in freq_counts.most_common(5)]
    else:
        primary_freqs = []

    all_freq_set = set(int(round(f)) for f in all_tonals)

    # Remarks - clean ASCII
    remarks_parts = [ev['assessment']]
    if ev.get('demon_confidence') and ev['demon_confidence'] != 'NONE':
        remarks_parts.append(f'DEMON: {ev["demon_confidence"]} confidence')
        if ev.get('est_n_blades') and ev.get('est_rpm'):
            remarks_parts.append(f'{ev["est_n_blades"]}-blade {ev["est_rpm"]:.0f} RPM')
    remarks_parts.append(f'{ev["rms_ratio"]:.0f}x background')
    if ev.get('demon_confidence') == 'NONE' and ev['rms_ratio'] >= SIGNIFICANT_RMS_FACTOR:
        remarks_parts.append('DEMON: not attempted (insufficient tonals)')
    remarks = '; '.join(remarks_parts)

    n_rows = max(len(primary_freqs), 1)

    # Date (only on change)
    if date_str != current_date:
        ws.cell(row=row, column=1, value=date_str).font = data_font
        ws.cell(row=row, column=1).alignment = left_align
        current_date = date_str

    # Time, Event, Remarks on first row
    ws.cell(row=row, column=2, value=time_str).font = data_font
    ws.cell(row=row, column=2).alignment = left_align
    ws.cell(row=row, column=12, value=ev['assessment']).font = data_font
    ws.cell(row=row, column=12).alignment = left_align
    ws.cell(row=row, column=13, value=remarks).font = data_font
    ws.cell(row=row, column=13).alignment = left_align

    # DEMON on first row
    if ev.get('blade_rate_hz') and ev['demon_confidence'] != 'NONE':
        ws.cell(row=row, column=10, value=round(ev['blade_rate_hz'], 1)).font = data_font
        ws.cell(row=row, column=10).alignment = center
    if ev.get('shaft_rate_hz') and ev['demon_confidence'] != 'NONE':
        ws.cell(row=row, column=11, value=round(ev['shaft_rate_hz'], 2)).font = data_font
        ws.cell(row=row, column=11).alignment = center

    # Each primary frequency on its own row
    for fi in range(n_rows):
        r = row + fi
        if fi < len(primary_freqs):
            freq_int = int(round(primary_freqs[fi]))

            # Spectrogram column
            ws.cell(row=r, column=3, value=freq_int).font = data_font
            ws.cell(row=r, column=3).alignment = center

            # Harmonics: check 2x and 3x
            h2 = freq_int * 2
            h3 = freq_int * 3
            h2_ok = h2 in all_freq_set
            h3_ok = h3 in all_freq_set
            if h2_ok and h3_ok:
                harm = f'{h2}, {h3}'
            elif h2_ok:
                harm = str(h2)
            elif h3_ok:
                harm = str(h3)
            else:
                harm = 'Not Present'
            ws.cell(row=r, column=4, value=harm).font = data_font
            ws.cell(row=r, column=4).alignment = center

            # LOFAR columns (same data)
            ws.cell(row=r, column=8, value=freq_int).font = data_font
            ws.cell(row=r, column=8).alignment = center
            ws.cell(row=r, column=9, value=harm).font = data_font
            ws.cell(row=r, column=9).alignment = center

        # Borders on all cells
        for col in range(1, 14):
            ws.cell(row=r, column=col).border = thin_border

    row += n_rows

# Notes
row += 2
ws.cell(row=row, column=5, value='BTR = Bearing-Time Record').font = Font(size=8, italic=True, name='Arial')
row += 1
ws.cell(row=row, column=1, value='Notes:').font = Font(size=8, bold=True, name='Arial')
row += 1
ws.cell(row=row, column=1, value='Bearing features require multi-element array; single hydrophone deployment - no bearing data available.').font = Font(size=8, italic=True, name='Arial')
row += 1
ws.cell(row=row, column=1, value='DEMON blade rate / shaft rate: extracted via Hilbert envelope analysis at 512 Hz. Confidence noted in Remarks.').font = Font(size=8, italic=True, name='Arial')
row += 1
ws.cell(row=row, column=1, value='HIGH = harmonics confirmed + strong RMS + multiple tonals; LOW = gate passed but fewer confirming indicators.').font = Font(size=8, italic=True, name='Arial')

ws.freeze_panes = 'A3'

xlsx_path = os.path.join(OUTPUT_DIR, 'havs_goa_results.xlsx')
wb.save(xlsx_path)
print(f'Customer XLSX saved: {xlsx_path}')
print(f'Significant events: {sum(1 for e in events if e["assessment"] != "brief transient")}')


Customer XLSX saved: /content/drive/MyDrive/SKANN_SSL/lofar_results_28jan/havs_goa_results.xlsx
Significant events: 2


## 11. Final Summary

In [ ]:
print('='*60)
print('LOFAR + DEMON HYDROPHONE ANALYSIS — COMPLETE')
print('='*60)
print(f'\nData processed: {len(df)} windows from {df["session"].nunique()} files')
print(f'Date range: {df["ts"].min().strftime("%d %b %H:%M")} → {df["ts"].max().strftime("%d %b %H:%M")}')
print(f'\nStage 1 (RMS Detection):')
print(f'  Dynamic range: {df["rms_energy"].max()/df["rms_energy"].min():.0f}×')
print(f'  Elevated windows: {(df["rms_energy"] > rms_thresh).sum()} ({100*(df["rms_energy"] > rms_thresh).sum()/len(df):.1f}%)')
print(f'\nStage 2 (LOFAR Classification):')
print(f'  Windows with tonals: {(df["n_tonals"] > 0).sum()} ({100*(df["n_tonals"] > 0).sum()/len(df):.1f}%)')
print(f'\nStage 3 (DEMON Characterisation):')
demon_attempted = (df['demon_confidence'] != 'NONE').sum()
print(f'  DEMON attempted: {demon_attempted} windows ({100*demon_attempted/len(df):.1f}%)')
print(f'  HIGH confidence: {(df["demon_confidence"] == "HIGH").sum()} windows')
print(f'  LOW confidence:  {(df["demon_confidence"] == "LOW").sum()} windows')
print(f'\nEvent Summary:')
print(f'  Total events: {len(events)}')
print(f'  Significant transits: {len(significant)}')
confirmed = [e for e in significant if 'CONFIRMED' in e.get('assessment', '')]
probable = [e for e in significant if 'PROBABLE' in e.get('assessment', '')]
print(f'  Confirmed vessels: {len(confirmed)}')
print(f'  Probable vessels: {len(probable)}')
print(f'\nOutput files in {OUTPUT_DIR}:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    if not f.startswith('.') and not f.startswith('_'):
        fpath = os.path.join(OUTPUT_DIR, f)
        size = os.path.getsize(fpath)
        print(f'  {f:55s} {size/1024:>8.0f} KB')


LOFAR + DEMON HYDROPHONE ANALYSIS — COMPLETE

Data processed: 789 windows from 4 files
Date range: 28 Jan 13:25 → 28 Jan 16:43

Stage 1 (RMS Detection):
  Dynamic range: 21×
  Elevated windows: 63 (8.0%)

Stage 2 (LOFAR Classification):
  Windows with tonals: 517 (65.5%)

Stage 3 (DEMON Characterisation):
  DEMON attempted: 13 windows (1.6%)
  HIGH confidence: 7 windows
  LOW confidence:  6 windows

Event Summary:
  Total events: 3
  Significant transits: 2
  Confirmed vessels: 1
  Probable vessels: 1

Output files in /content/drive/MyDrive/SKANN_SSL/lofar_results_28jan/:
  havs_goa_results.xlsx                                          6 KB
  lofar_hydrophone_results.csv                                 160 KB
  lofar_session_28jan_1.png                                   2171 KB
  lofar_session_28jan_2.png                                   2189 KB
  lofar_session_28jan_3.png                                   2217 KB
  lofar_session_28jan_4.png                                   1642 KB
 